# tiny-jepa — World Labs for your laptop

A tiny, hackable, action-conditioned JEPA world model in pure numpy. Zero dependencies.
In this notebook: install it, watch the pretrained model hunt pellets from raw pixels,
peek inside its imagined futures, and train a fresh one.

Repo: https://github.com/OpenSourceRulesz/tiny-jepa

In [ ]:
!pip install -q git+https://github.com/OpenSourceRulesz/tiny-jepa.git
import numpy as np
from tiny_jepa import TinyJEPA, OrbyWorld
print("tiny-jepa", TinyJEPA is not None)

## 1. Watch it hunt

The shipped checkpoint perceives the pellet with learned vision and plans one step ahead.

In [ ]:
import os, tiny_jepa
ckpt = os.path.join(os.path.dirname(tiny_jepa.__file__), "checkpoints", "orby_v1.npz")
model = TinyJEPA.load(ckpt)
env = OrbyWorld(seed=7)
eaten = 0
for t in range(64):
    frame = env.render()
    pellet = model.perceive_pellet(frame)
    action = model.plan(frame, env.agent)
    _, r = env.step(action)
    eaten += int(r > 0.5)
    if t % 16 == 0:
        print(f"t={t:2d} pellet~{pellet.round(1)} true~{env.pellet.round(1)} eaten={eaten}")
print(f"\n{eaten} pellets / 64 steps  (random policy: ~0.2)")

## 2. Peek inside its imagination

`imagine` rolls the predictor forward in latent space; `dream` decodes those latents back to frames.

In [ ]:
import matplotlib.pyplot as plt
frame = env.render()
z = model.encode(frame)
dreams = model.dream(frame, [4, 4, 4])  # what if it goes right 3x?
fig, ax = plt.subplots(1, 4, figsize=(10, 3))
ax[0].imshow(frame, cmap="gray"); ax[0].set_title("reality")
for i, d in enumerate(dreams):
    ax[i+1].imshow(d, cmap="gray"); ax[i+1].set_title(f"dream t+{i+1}")
    ax[i+1].axis("off")
ax[0].axis("off"); plt.tight_layout(); plt.show()

## 3. Train your own (tiny demo — 60 steps, ~1 min)

Full training is 1500 steps (~8 min on CPU). This runs 60 steps so you can watch the loss fall.

In [ ]:
from tiny_jepa.train import train_on_env
mine = train_on_env(lambda seed: OrbyWorld(seed=seed), steps=60, batch=128, log_every=20)
print("trained! pellet perception check:")
f = env.render()
print("perceived:", mine.perceive_pellet(f).round(1), " true:", env.pellet.round(1))

## Go crazy

The `challenges/` folder in the repo has 7 quests — beat the greedy planner, sharpen the dreams,
shrink the model, bring your own world. The leaderboard is empty. Your move.